In [25]:

!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

In [26]:
from sentence_transformers import SentenceTransformer

# Load the sentence-transformer embedding model
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Embedding model successfully loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model successfully loaded!


In [27]:

GEMMA_MODEL = "gemma-4-26b-a4b-it"

CHUNK_SIZE = 120

CHUNK_OVERLAP = 30

TOP_K = 3

ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [28]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model where the trained parameters (weights) are made publicly available, allowing developers to run, study, and fine-tune the model on their own hardware.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [29]:
# Academic Database mapping all 10 standard university compliance metrics
SAMPLE_PAGES = [
    """[Page 1] College Profile:
    Institution Name: Apex Institute of Technology (AIT).
    Address: 742 Science & Innovation Boulevard, Tech District, Sector 4.
    Contact: info@ait.edu | +1 (555) 019-9800.
    Website: https://www.ait.edu
    Affiliation: Fully affiliated with the National Technological University (NTU) and accredited by the Global Board of Engineering and Applied Sciences with an A++ rating.""",

    """[Page 2] Courses and Departments:
    Branches: Computer Science & Engineering (CSE), Electronics & Communication (ECE), Mechanical Engineering (ME).
    Primary Course: Bachelor of Technology (B.Tech).
    Subject Codes & Syllabus:
    - CS-101: Introduction to Programming (Python variables, arrays, loops, API design, file operations).
    - MATH-201: Linear Algebra & Matrices (Vector spaces, linear systems, eigenvalues, transformations).
    - PHYS-102: General Physics II (Electromagnetism, circuit laws, Maxwell equations, quantum fundamentals).
    - ENG-115: Professional Technical Writing (Technical reports, project proposals, documentation standards).""",

    """[Page 3] Attendance Rules:
    Minimum Attendance Policy: Students must maintain a minimum of 75% overall attendance to qualify for the end-semester final exams.
    Shortage and Warnings: Attendance is monitored bi-weekly. Warning notices are sent to students falling below 80%.
    Condonation: Students with attendance between 65% and 74% due to medical emergencies can apply for condonation by submitting medical certificates to the Dean of Academic Affairs along with a processing fee of $50. Attendance under 65% results in automatic exam ineligibility.""",

    """[Page 4] Examination Rules:
    Exam Eligibility: Must satisfy both the 75% minimum attendance rule and have cleared all outstanding university dues.
    Passing Marks: To pass any course, a student must secure at least 40% in the Semester End Examination (SEE) and a combined total (CIE + SEE) of 50%.
    Academic Malpractice Policy: Possession of unauthorized papers, digital devices, or consulting peers during examinations leads to immediate suspension from all current-semester exams and disciplinary hearing review.""",

    """[Page 5] Internal Assessment:
    Continuous Internal Evaluation (CIE): Accounts for 30% of the aggregate grade.
    CIE Breakdowns: Two midterm tests (15% total), classroom assignments (10%), and class participation & quizzes (5%).
    Test Schedules: Midterm Test 1 takes place in Week 6; Midterm Test 2 takes place in Week 12. Assignments must be submitted via the student portal; late submissions lose 10% score per day.""",

    """[Page 6] Results and Revaluation:
    Result Announcements: Final course grades are published on the official Student Portal within 21 days after the final examination concludes.
    Revaluation Procedure: Dissatisfied students can file for digital revaluation or answer script copy requests within 10 days of results release.
    Revaluation Fees: $40 per course script.
    Supplementary Exams: Conducted annually during the summer semester for students who failed to achieve passing grades.""",

    """[Page 7] Academic Calendar:
    Semester Duration: Fall 2024 begins August 26, 2024, and ends December 18, 2024.
    Holidays: Labor Day (Sept 2), Thanksgiving Break (Nov 25-29), Winter Break starts Dec 19.
    Important Deadlines: Last day to add/drop courses is September 6. Course registration for Spring 2025 opens November 12. End-Semester Examination starts December 5.""",

    """[Page 8] Student Facilities:
    Library: Open 24/7, houses over 150,000 reference volumes, and grants full digital access to IEEE, ACM, and ScienceDirect journals.
    Laboratories: High-performance computer labs with NVIDIA GPUs, advanced physics labs, and rapid prototyping fabrication workshops.
    Other Facilities: Student housing (Hostel-A & Hostel-B), 24/7 campus shuttle bus service, dynamic tech canteen, and recreational athletic center.""",

    """[Page 9] Fees and Scholarships:
    Tuition Fee Schedule: B.Tech tuition is $8,500 per semester, payable by August 20 for Fall and January 15 for Spring.
    Scholarship Eligibility:
    - Merit Scholar: 100% tuition waiver for students maintaining a CGPA of 3.90 or above.
    - Achievement Award: 50% tuition waiver for students with a CGPA between 3.75 and 3.89.
    Application Deadline: All scholarship renewal forms must be submitted before June 30 annually.""",

    """[Page 10] Contacts and Support:
    Academic Office: Room A-102 | academic.support@ait.edu | Ext. 401.
    IT Help Desk: Support for logins, campus Wi-Fi, and software licenses | helpdesk@ait.edu | Ext. 101.
    Student Welfare & Counseling: Safe counseling space and mental health services | welfare@ait.edu | Ext. 305.
    Dean of Academic Affairs Office: Dean Dr. Raymond Vance | dean.office@ait.edu."""
]
print(f"Academic Database populated with {len(SAMPLE_PAGES)} detailed page records.")

Academic Database populated with 10 detailed page records.


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [30]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 10 pages, 10 chunks.

Question: What happens if I submit a lab late?
Answer: The answer could not be found in the document.

Retrieved passages:
   [page 6, similarity 0.36] [Page 6] Results and Revaluation: Result Announcements: Final course grades are published ...
   [page 4, similarity 0.28] [Page 4] Examination Rules: Exam Eligibility: Must satisfy both the 75% minimum attendance...
   [page 7, similarity 0.25] [Page 7] Academic Calendar: Semester Duration: Fall 2024 begins August 26, 2024, and ends ...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [39]:
import os
import gradio as gr

gr.close_all()  # Close active servers

APP_TITLE = "🎓 NEXUS: Elite Student Academic Portal"
APP_DESCRIPTION = "Welcome to the next-generation academic portal. Access authenticated credentials, track overall transcript metrics, and consult with our hyper-accurate, 10-domain trained RAG engine for real-time compliance information."

USER_CREDENTIALS = {
    "alex": "password123",
    "taylor": "secure456"
}

STUDENT_DATA = {
    "name": "Alex Morgan",
    "id": "#2024-89745",
    "major": "Computer Science & Engineering",
    "semester": "Fall 2024 (Sophomore)",
    "cgpa": "3.82",
    "overall_percentage": "91.5%",
    "overall_attendance": "92%",
    "overall_status": "EXCELLENT",
    "backlog_details": "0 (No Active Backlogs)",
    "advisor": "Dr. Amanda Rao",
    "department": "School of Computer Science & Technology",
    "subjects": [
        {
            "code": "CS-101",
            "name": "Introduction to Programming",
            "attendance_pct": 95,
            "internals": "27 / 30",
            "externals": "63 / 70",
            "total": 90,
            "status": "PASS"
        },
        {
            "code": "MATH-201",
            "name": "Linear Algebra & Matrices",
            "attendance_pct": 88,
            "internals": "24 / 30",
            "externals": "58 / 70",
            "total": 82,
            "status": "PASS"
        },
        {
            "code": "PHYS-102",
            "name": "General Physics II (Electromagnetism)",
            "attendance_pct": 90,
            "internals": "26 / 30",
            "externals": "60 / 70",
            "total": 86,
            "status": "PASS"
        },
        {
            "code": "ENG-115",
            "name": "Professional Technical Writing",
            "attendance_pct": 94,
            "internals": "29 / 30",
            "externals": "65 / 70",
            "total": 94,
            "status": "PASS"
        }
    ]
}

def authenticate_student(username, password):
    username_clean = username.strip().lower()
    if username_clean in USER_CREDENTIALS and USER_CREDENTIALS[username_clean] == password:
        return (
            gr.update(visible=False),
            gr.update(visible=True),
            ""
        )
    else:
        return (
            gr.update(visible=True),
            gr.update(visible=False),
            "❌ Authentication Failed! Try 'alex' / 'password123'"
        )

def on_upload(file):
    if file is None:
        return "No file uploaded."
    try:
        return build_knowledge_base(read_pdf(file), os.path.basename(file))
    except Exception as e:
        raise gr.Error(f"The PDF could not be read: {e}")

def on_ask(question):
    if not question.strip():
        raise gr.Error("Please ask a valid question.")
    reply, found = answer(question)
    sources = ""
    for i, c in enumerate(found, start=1):
        sources += f"<div class='citation-box'><strong>Passage {i} (Page {c['page']} | Match Score: {c['score']:.2f})</strong><br><p>{c['text']}</p></div>"
    return reply, sources

def render_academic_portal():
    rows_html = ""
    for sub in STUDENT_DATA["subjects"]:
        att_color = "#10b981" if sub["attendance_pct"] >= 90 else "#f59e0b"
        status_badge = "<span style='background-color: #d1fae5; color: #065f46; padding: 4px 10px; border-radius: 9999px; font-size: 0.75rem; font-weight: bold;'>PASS</span>"

        rows_html += f"""
        <tr style='border-bottom: 1px solid #f1f5f9; transition: background-color 0.2s ease;'>
            <td style='padding: 16px 12px; font-weight: 700; color: #1e293b;'>{sub['code']}</td>
            <td style='padding: 16px 12px; color: #334155; font-weight: 500;'>{sub['name']}</td>
            <td style='padding: 16px 12px;'>
                <div style='display: flex; align-items: center; gap: 8px;'>
                    <div style='background-color: #e2e8f0; width: 60px; height: 6px; border-radius: 9999px; overflow: hidden;'>
                        <div style='background-color: {att_color}; width: {sub['attendance_pct']}%; height: 100%;'></div>
                    </div>
                    <span style='font-size: 0.85rem; font-weight: 700; color: #1e293b;'>{sub['attendance_pct']}%</span>
                </div>
            </td>
            <td style='padding: 16px 12px; text-align: center; color: #475569;'>{sub['internals']}</td>
            <td style='padding: 16px 12px; text-align: center; color: #475569;'>{sub['externals']}</td>
            <td style='padding: 16px 12px; text-align: center; font-weight: 700; color: #0f172a;'>{sub['total']} / 100</td>
            <td style='padding: 16px 12px; text-align: center;'>{status_badge}</td>
        </tr>
        """

    dashboard_html = f"""
    <div style='background-color: #ffffff; border-radius: 16px; font-family: system-ui, -apple-system, sans-serif;'>
        <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 20px; margin-bottom: 30px;'>
            <div style='background: linear-gradient(135deg, #eff6ff 0%, #dbeafe 100%); border: 1px solid #bfdbfe; padding: 20px; border-radius: 14px; position: relative; overflow: hidden;'>
                <div style='position: absolute; right: -15px; bottom: -15px; font-size: 80px; opacity: 0.1; font-weight: 900; color: #1e3a8a;'>GPA</div>
                <span style='color: #1e40af; font-size: 0.8rem; text-transform: uppercase; font-weight: 800; letter-spacing: 0.05em;'>Academic Standing</span>
                <div style='font-size: 2rem; font-weight: 800; color: #1e3a8a; margin-top: 6px;'>{STUDENT_DATA['cgpa']} CGPA</div>
                <span style='color: #2563eb; font-size: 0.9rem; font-weight: 600;'>Overall Score: {STUDENT_DATA['overall_percentage']}</span>
            </div>
            <div style='background: linear-gradient(135deg, #ecfdf5 0%, #d1fae5 100%); border: 1px solid #a7f3d0; padding: 20px; border-radius: 14px; position: relative; overflow: hidden;'>
                <div style='position: absolute; right: -15px; bottom: -15px; font-size: 80px; opacity: 0.1; font-weight: 900; color: #065f46;'>ATT</div>
                <span style='color: #065f46; font-size: 0.8rem; text-transform: uppercase; font-weight: 800; letter-spacing: 0.05em;'>Presence Rating</span>
                <div style='font-size: 2rem; font-weight: 800; color: #065f46; margin-top: 6px;'>{STUDENT_DATA['overall_attendance']}</div>
                <span style='color: #059669; font-size: 0.9rem; font-weight: 600;'>Status: Satisfactory</span>
            </div>
            <div style='background: linear-gradient(135deg, #faf5ff 0%, #f3e8ff 100%); border: 1px solid #e9d5ff; padding: 20px; border-radius: 14px; position: relative; overflow: hidden;'>
                <div style='position: absolute; right: -15px; bottom: -15px; font-size: 80px; opacity: 0.1; font-weight: 900; color: #581c87;'>REG</div>
                <span style='color: #581c87; font-size: 0.8rem; text-transform: uppercase; font-weight: 800; letter-spacing: 0.05em;'>Pending Backlogs</span>
                <div style='font-size: 2rem; font-weight: 800; color: #581c87; margin-top: 6px;'>{STUDENT_DATA['backlog_details']}</div>
                <span style='color: #7c3aed; font-size: 0.9rem; font-weight: 600;'>Good Academic Standing</span>
            </div>
        </div>
        <div style='border-bottom: 2px solid #f1f5f9; padding-bottom: 12px; margin-bottom: 20px; display: flex; justify-content: space-between; align-items: center;'>
            <h3 style='color: #0f172a; margin: 0; font-size: 1.4rem; font-weight: 800;'>📋 Certified Academic Transcript</h3>
            <span style='font-size: 0.85rem; background-color: #f1f5f9; padding: 6px 12px; border-radius: 6px; font-weight: 600; color: #475569;'>Term: {STUDENT_DATA['semester']}</span>
        </div>
        <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 25px; background-color: #f8fafc; padding: 16px; border-radius: 10px; border: 1px solid #e2e8f0;'>
            <div><strong style='color: #64748b; font-size: 0.8rem; text-transform: uppercase;'>Institution</strong><br><span style='font-weight: 600; color: #334155; font-size: 0.95rem;'>{STUDENT_DATA['department']}</span></div>
            <div><strong style='color: #64748b; font-size: 0.8rem; text-transform: uppercase;'>Assigned Advisor</strong><br><span style='font-weight: 600; color: #334155; font-size: 0.95rem;'>{STUDENT_DATA['advisor']}</span></div>
            <div><strong style='color: #64748b; font-size: 0.8rem; text-transform: uppercase;'>Current Program</strong><br><span style='font-weight: 600; color: #334155; font-size: 0.95rem;'>{STUDENT_DATA['major']} Degree</span></div>
        </div>
        <div style='overflow-x: auto; border-radius: 12px; border: 1px solid #f0f3f6;'>
            <table style='width: 100%; border-collapse: collapse; text-align: left; min-width: 700px;'>
                <thead>
                    <tr style='background-color: #f8fafc; border-bottom: 2px solid #e2e8f0;'>
                        <th style='padding: 14px 12px; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Code</th>
                        <th style='padding: 14px 12px; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Subject Title</th>
                        <th style='padding: 14px 12px; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Attendance Tracker</th>
                        <th style='padding: 14px 12px; text-align: center; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Internals (30)</th>
                        <th style='padding: 14px 12px; text-align: center; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Semester End (70)</th>
                        <th style='padding: 14px 12px; text-align: center; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Total (100)</th>
                        <th style='padding: 14px 12px; text-align: center; color: #475569; font-weight: 700; font-size: 0.85rem; text-transform: uppercase;'>Status</th>
                    </tr>
                </thead>
                <tbody>
                    {rows_html}
                </tbody>
            </table>
        </div>
    </div>
    """
    return dashboard_html

custom_css = """
.gradio-container {
    max-width: 1280px !important;
    margin: auto !important;
    padding: 40px 20px !important;
    background-color: #f6f8fc !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif !important;
}
.beauty-card {
    border: 1px solid #eef2f6 !important;
    border-radius: 20px !important;
    padding: 30px !important;
    background: #ffffff !important;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.02), 0 8px 10px -6px rgba(0, 0, 0, 0.02) !important;
    transition: transform 0.4s cubic-bezier(0.16, 1, 0.3, 1), box-shadow 0.4s cubic-bezier(0.16, 1, 0.3, 1), border-color 0.4s ease !important;
}
.beauty-card:hover {
    transform: translateY(-3px) !important;
    box-shadow: 0 20px 35px -5px rgba(37, 99, 235, 0.08), 0 10px 15px -5px rgba(37, 99, 235, 0.03) !important;
    border-color: #2563eb !important;
}
.login-card {
    max-width: 480px;
    margin: 60px auto !important;
    border: 1px solid #e2e8f0 !important;
    border-radius: 24px !important;
    padding: 40px !important;
    background: #ffffff !important;
    box-shadow: 0 20px 40px -15px rgba(15, 23, 42, 0.1) !important;
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important;
}
.header-box {
    background: linear-gradient(135deg, #1e3a8a 0%, #0f172a 100%) !important;
    padding: 40px !important;
    border-radius: 24px !important;
    color: #ffffff !important;
    margin-bottom: 30px !important;
    box-shadow: 0 20px 40px -10px rgba(30, 58, 138, 0.3) !important;
}
.header-box h1 {
    color: #ffffff !important;
    font-size: 2.5rem !important;
    font-weight: 900 !important;
    margin-bottom: 10px !important;
}
.header-box p {
    color: #cbd5e1 !important;
    font-size: 1.15rem !important;
    line-height: 1.6 !important;
}
.header-kb-source {
    text-align: right !important;
    color: #ffffff !important;
}
.header-kb-source p, .header-kb-source span, .header-kb-source div {
    color: #ffffff !important;
}
.answer-card {
    border: 1px solid #dbeafe !important;
    border-left: 6px solid #2563eb !important;
    background: #f8fafc !important;
    border-radius: 12px !important;
    padding: 24px !important;
}
.citation-box {
    background-color: #fafbfd;
    border: 1px solid #e4ecf3;
    border-radius: 10px;
    padding: 16px;
    margin-bottom: 12px;
}
button {
    border-radius: 12px !important;
    padding: 12px 24px !important;
    font-weight: 700 !important;
    transition: all 0.2s ease-in-out !important;
}
button.primary {
    background: linear-gradient(135deg, #2563eb 0%, #1d4ed8 100%) !important;
    color: white !important;
}
button.primary:hover {
    transform: scale(1.02) !important;
    box-shadow: 0 8px 20px rgba(37, 99, 235, 0.3) !important;
}
input, textarea {
    border-radius: 12px !important;
    border: 1px solid #e2e8f0 !important;
    transition: border-color 0.25s ease, box-shadow 0.25s ease !important;
}
input:focus, textarea:focus {
    border-color: #2563eb !important;
    box-shadow: 0 0 0 4px rgba(37, 99, 235, 0.1) !important;
}
"""

with gr.Blocks(title="NEXUS Portal") as demo:

    # ================= LOGIN INTERFACE =================
    with gr.Column(elem_classes="login-card", visible=True) as login_container:
        gr.HTML("""
        <div style='text-align: center; margin-bottom: 24px;'>
            <div style='font-size: 40px; margin-bottom: 10px;'>🎓</div>
            <h2 style='font-size: 1.8rem; font-weight: 800; color: #0f172a; margin: 0;'>NEXUS STUDENT PORTAL</h2>
            <p style='color: #64748b; font-size: 0.95rem; margin-top: 6px;'>Please authenticate using your credentials to enter the workspace.</p>
        </div>
        """)

        username_input = gr.Textbox(label="Username / Student ID", placeholder="e.g., alex", max_lines=1)
        password_input = gr.Textbox(label="Password", placeholder="••••••••", type="password", max_lines=1)
        login_btn = gr.Button("Secure Sign In", variant="primary")
        login_error = gr.Markdown(value="", elem_id="login-error")

        gr.HTML("""
        <div style='margin-top: 20px; padding: 12px; background-color: #f8fafc; border-radius: 8px; border: 1px dashed #cbd5e1; text-align: center;'>
            <span style='font-size: 0.8rem; color: #64748b;'><strong>Demo Accounts Available:</strong><br>Username: <code>alex</code> | Password: <code>password123</code></span>
        </div>
        """)

    # ================= SECURED MAIN WEBSITE INTERFACE =================
    with gr.Column(visible=False) as main_portal_container:

        # Beautiful premium layout header
        with gr.Row(elem_classes="header-box"):
            with gr.Column(scale=4):
                gr.Markdown(f"# {APP_TITLE}")
                gr.Markdown(f"{APP_DESCRIPTION}")
            with gr.Column(scale=1, min_width=180, elem_classes="header-kb-source"):
                status_md = gr.Markdown(f"📜 **Knowledge Base Source:**\n`{knowledge_base['name'] or 'None'}`")

        with gr.Tabs():

            # TAB 1: ACADEMIC TRANSCRIPT & METRICS
            with gr.TabItem("🎓 Academic Hub", id="performance_tab"):
                with gr.Column(elem_classes="beauty-card"):
                    gr.HTML(value=render_academic_portal())

            # TAB 2: SMART DOCUMENT CONSULTANT & AI RAG CHAT
            with gr.TabItem("💬 AI Document Assistant (RAG)", id="rag_tab"):
                with gr.Row(equal_height=True):
                    with gr.Column(scale=1, elem_classes="beauty-card"):
                        gr.Markdown("### 📁 1. Document Management")
                        pdf_input = gr.File(label="Upload Academic PDF", file_types=[".pdf"], type="filepath")
                        sample_btn = gr.Button("💡 Auto-Load Integrated Academic Database", variant="secondary")

                    with gr.Column(scale=2, elem_classes="beauty-card"):
                        gr.Markdown("### 🗣️ 2. Interactive Consultation")
                        question_box = gr.Textbox(
                            label="Ask your question",
                            placeholder="Ask about attendance shortage, exams, passing marks, revaluation fee, etc.",
                            max_lines=2
                        )
                        with gr.Row():
                            ask_btn = gr.Button("✨ Submit Search & Query", variant="primary")

                # Suggested Queries
                gr.Examples(
                    examples=[
                        "What is the minimum attendance requirement and the rules for shortage?",
                        "What are the rules, eligibility, and passing marks for examinations?",
                        "What are the scholarship criteria and the application deadline?",
                        "How do I request revaluation and what is the fee per script?",
                        "What is the address, site, and affiliation of the college?"
                    ],
                    inputs=question_box,
                    label="💡 Recommended Sample Compliance Questions"
                )

                gr.HTML("<br>")

                with gr.Column(elem_classes="beauty-card"):
                    gr.Markdown("### 🏉 AI Response & Source Verification")
                    with gr.Column(elem_classes="answer-card"):
                        answer_md = gr.Markdown(value="*Awaiting your question...*", label="Answer")
                    gr.HTML("<br>")
                    with gr.Accordion("🔍 View Context References & Similarity Metrics", open=False):
                        sources_html = gr.HTML("<p style='color: #64748b;'>No context chunks fetched yet.</p>")

    # Event Bindings
    login_btn.click(
        authenticate_student,
        inputs=[username_input, password_input],
        outputs=[login_container, main_portal_container, login_error]
    )
    password_input.submit(
        authenticate_student,
        inputs=[username_input, password_input],
        outputs=[login_container, main_portal_container, login_error]
    )

    pdf_input.upload(on_upload, inputs=pdf_input, outputs=status_md)
    sample_btn.click(lambda: build_knowledge_base(SAMPLE_PAGES, "Upgraded Academic Database"), outputs=status_md)

    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_html])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_html])

Closing server running on port: 7860


In [32]:
# Launch the integrated 10/10 compliance and academic hub
build_knowledge_base(SAMPLE_PAGES, "Integrated Academic Database")
demo.launch(
    share=True,
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
    css=custom_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f487a329915652791.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
# Programmatic verification test for the NEXUS secure login system
test_user = 'alex'
test_pass = 'password123'

# Run the authentication function used by the Gradio app
login_hidden, portal_visible, error_message = authenticate_student(test_user, test_pass)

print("=== NEXUS LOGIN PORTAL TEST ===")
print(f"Testing Username: '{test_user}'")
print(f"Testing Password: '{test_pass}'")
print("-------------------------------")
print(f"Action - Hide Login Card: {not login_hidden['visible']}")
print(f"Action - Show Main Dashboard: {portal_visible['visible']}")
print(f"Action - Error Message Displayed: '{error_message or 'None - Success'}'")


=== NEXUS LOGIN PORTAL TEST ===
Testing Username: 'alex'
Testing Password: 'password123'
-------------------------------
Action - Hide Login Card: True
Action - Show Main Dashboard: True
Action - Error Message Displayed: 'None - Success'


In [34]:
# Re-initialize the active knowledge base with all 10 compliance areas
status_msg = build_knowledge_base(SAMPLE_PAGES, 'Academic Database')
print(status_msg)
print("Database matches all 10 academic domains:")
for i, doc in enumerate(SAMPLE_PAGES, start=1):
    first_line = doc.strip().split('\n')[0]
    print(f"  Domain {i:02d}: {first_line}")

Loaded **Academic Database**: 10 pages, 10 chunks.
Database matches all 10 academic domains:
  Domain 01: [Page 1] College Profile:
  Domain 02: [Page 2] Courses and Departments:
  Domain 03: [Page 3] Attendance Rules:
  Domain 04: [Page 4] Examination Rules:
  Domain 05: [Page 5] Internal Assessment:
  Domain 06: [Page 6] Results and Revaluation:
  Domain 07: [Page 7] Academic Calendar:
  Domain 08: [Page 8] Student Facilities:
  Domain 09: [Page 9] Fees and Scholarships:
  Domain 10: [Page 10] Contacts and Support:


In [35]:
def on_upload_multiple(files):
    """Handles uploading multiple PDFs, combines all extracted pages, and rebuilds the knowledge base."""
    if not files:
        return "No files uploaded."

    combined_pages = []
    file_names = []

    # Process each uploaded file
    for file_obj in files:
        try:
            filename = os.path.basename(file_obj.name)
            file_names.append(filename)

            # Extract pages and attach document attribution to each page
            reader = PdfReader(file_obj.name)
            for i, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ""
                # Embed the document name directly inside the page chunk header for accurate citation
                combined_pages.append(f"[Document: {filename} | Page {i}] {text}")
        except Exception as e:
            raise gr.Error(f"Failed to process PDF file '{filename}': {e}")

    db_name = ", ".join(file_names)
    # Rebuild vectors for all uploaded pages
    return build_knowledge_base(combined_pages, f"Multi-Doc: {db_name}")

In [40]:
# Fully completed and integrated Gradio interface incorporating multiple PDF upload support
import gradio as gr

# Fallback configurations if setup cell wasn't executed in current memory session
if "APP_TITLE" not in globals():
    APP_TITLE = "🎓 NEXUS: Elite Student Academic Portal"
if "APP_DESCRIPTION" not in globals():
    APP_DESCRIPTION = "Welcome to the next-generation academic portal. Access authenticated credentials, track overall transcript metrics, and consult with our hyper-accurate, 10-domain trained RAG engine for real-time compliance information."
if "knowledge_base" not in globals():
    knowledge_base = {"name": None, "chunks": [], "vectors": None}

with gr.Blocks(title="NEXUS Portal") as demo:

    # ================= LOGIN INTERFACE =================
    with gr.Column(elem_classes="login-card", visible=True) as login_container:
        gr.HTML("""
        <div style='text-align: center; margin-bottom: 24px;'>
            <div style='font-size: 40px; margin-bottom: 10px;'>🎓</div>
            <h2 style='font-size: 1.8rem; font-weight: 800; color: #0f172a; margin: 0;'>NEXUS STUDENT PORTAL</h2>
            <p style='color: #64748b; font-size: 0.95rem; margin-top: 6px;'>Please authenticate using your credentials to enter the workspace.</p>
        </div>
        """)

        username_input = gr.Textbox(label="Username / Student ID", placeholder="e.g., alex", max_lines=1)
        password_input = gr.Textbox(label="Password", placeholder="••••••••", type="password", max_lines=1)
        login_btn = gr.Button("Secure Sign In", variant="primary")
        login_error = gr.Markdown(value="", elem_id="login-error")

        gr.HTML("""
        <div style='margin-top: 20px; padding: 12px; background-color: #f8fafc; border-radius: 8px; border: 1px dashed #cbd5e1; text-align: center;'>
            <span style='font-size: 0.8rem; color: #64748b;'><strong>Demo Accounts Available:</strong><br>Username: <code>alex</code> | Password: <code>password123</code></span>
        </div>
        """)

    # ================= SECURED MAIN WEBSITE INTERFACE =================
    with gr.Column(visible=False) as main_portal_container:

        with gr.Row(elem_classes="header-box"):
            with gr.Column(scale=4):
                gr.Markdown(f"# {APP_TITLE}")
                gr.Markdown(f"{APP_DESCRIPTION}")
            with gr.Column(scale=1, min_width=180, elem_classes="header-kb-source"):
                status_md = gr.Markdown(f"📜 **Knowledge Base Source:**\n`{knowledge_base['name'] or 'None'}`")

        with gr.Tabs():
            # TAB 1: ACADEMIC TRANSCRIPT & METRICS
            with gr.TabItem("🎓 Academic Hub", id="performance_tab"):
                with gr.Column(elem_classes="beauty-card"):
                    gr.HTML(value=render_academic_portal() if 'render_academic_portal' in globals() else "<p>Academic hub transcript generator not loaded.</p>")

            # TAB 2: SMART DOCUMENT CONSULTANT & AI RAG CHAT
            with gr.TabItem("💬 AI Document Assistant (RAG)", id="rag_tab"):
                with gr.Row(equal_height=True):
                    with gr.Column(scale=1, elem_classes="beauty-card"):
                        gr.Markdown("### 📁 1. Document Management")
                        pdf_input = gr.File(label="Upload Academic PDFs", file_types=[".pdf"], file_count="multiple", type="filepath")
                        sample_btn = gr.Button("💡 Auto-Load Integrated Academic Database", variant="secondary")

                    with gr.Column(scale=2, elem_classes="beauty-card"):
                        gr.Markdown("### 🗣️ 2. Interactive Consultation")
                        question_box = gr.Textbox(
                            label="Ask your question",
                            placeholder="Ask about attendance shortage, exams, passing marks, revaluation fee, etc.",
                            max_lines=2
                        )
                        with gr.Row():
                            ask_btn = gr.Button("✨ Submit Search & Query", variant="primary")

                gr.Examples(
                    examples=[
                        "What is the minimum attendance requirement and the rules for shortage?",
                        "What are the rules, eligibility, and passing marks for examinations?",
                        "What are the scholarship criteria and the application deadline?",
                        "How do I request revaluation and what is the fee per script?",
                        "What is the address, site, and affiliation of the college?"
                    ],
                    inputs=question_box,
                    label="💡 Recommended Sample Compliance Questions"
                )

                gr.HTML("<br>")

                with gr.Column(elem_classes="beauty-card"):
                    gr.Markdown("### 🏉 AI Response & Source Verification")
                    with gr.Column(elem_classes="answer-card"):
                        answer_md = gr.Markdown(value="*Awaiting your question...*", label="Answer")
                    gr.HTML("<br>")
                    with gr.Accordion("🔍 View Context References & Similarity Metrics", open=False):
                        sources_html = gr.HTML("<p style='color: #64748b;'>No context chunks fetched yet.</p>")

    # Event Bindings
    login_btn.click(
        authenticate_student if 'authenticate_student' in globals() else lambda u, p: (gr.update(), gr.update(), "Auth function missing"),
        inputs=[username_input, password_input],
        outputs=[login_container, main_portal_container, login_error]
    )
    password_input.submit(
        authenticate_student if 'authenticate_student' in globals() else lambda u, p: (gr.update(), gr.update(), "Auth function missing"),
        inputs=[username_input, password_input],
        outputs=[login_container, main_portal_container, login_error]
    )

    # Integrated with our multiple PDF handler function
    pdf_input.upload(on_upload_multiple if 'on_upload_multiple' in globals() else lambda f: "Handler missing", inputs=pdf_input, outputs=status_md)
    sample_btn.click(lambda: build_knowledge_base(SAMPLE_PAGES, "Academic Database") if 'build_knowledge_base' in globals() else "Loader missing", outputs=status_md)

    ask_btn.click(on_ask if 'on_ask' in globals() else lambda q: ("Handler missing", ""), inputs=question_box, outputs=[answer_md, sources_html])
    question_box.submit(on_ask if 'on_ask' in globals() else lambda q: ("Handler missing", ""), inputs=question_box, outputs=[answer_md, sources_html])

In [37]:
# Re-launch the integrated multi-PDF RAG student portal
gr.close_all()
build_knowledge_base(SAMPLE_PAGES, "Academic Database")
demo.launch(
    share=True,
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
    css=custom_css
)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a28fb0ea020ee70266.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [38]:
# Final End-to-End Validation Test
import sys

print("=== RUNNING FINAL PIPELINE VERIFICATION ===")

# Ensure prerequisite functions exist in global scope
if 'build_knowledge_base' not in globals() or 'answer' not in globals():
    print("[WARNING] Pipeline functions are not initialized in memory.")
    print("Please run all cells above (Runtime -> Run all) to initialize the RAG backend first.")
else:
    try:
        # 1. Verify Knowledge Base Initialization
        status = build_knowledge_base(SAMPLE_PAGES, "Final Verified Academic Database")
        print(f"[PASS] Knowledge Base Status: {status}")

        # 2. Test RAG Retrieval and Matching
        test_query = "What is the minimum attendance requirement?"
        results = retrieve(test_query, top_k=1)
        print(f"[PASS] Retrieval Test: Top match on Page {results[0]['page']} with Similarity Score: {results[0]['score']:.4f}")

        # 3. Test LLM Answer Generation
        reply, found = answer(test_query)
        print(f"[PASS] Generation Test:\nQuery: '{test_query}'\nResponse: '{reply}'")
        print("===========================================")
        print("STATUS: ALL SYSTEMS OPERATIONAL")
    except Exception as e:
        print(f"[FAIL] Pipeline test encountered an error: {e}")

=== RUNNING FINAL PIPELINE VERIFICATION ===
[PASS] Knowledge Base Status: Loaded **Final Verified Academic Database**: 10 pages, 10 chunks.
[PASS] Retrieval Test: Top match on Page 3 with Similarity Score: 0.6662
[PASS] Generation Test:
Query: 'What is the minimum attendance requirement?'
Response: 'Students must maintain a minimum of 75% overall attendance to qualify for the end-semester final exams [Page 3]. Additionally, students must satisfy this 75% rule to be eligible for examinations [Page 4].'
STATUS: ALL SYSTEMS OPERATIONAL


In [41]:
gr.close_all()
build_knowledge_base(SAMPLE_PAGES, 'Academic Database')
demo.launch(
    share=True,
    theme=gr.themes.Soft(primary_hue='blue', secondary_hue='slate'),
    css=custom_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d31624dec450772d63.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |